# Kapitel 12: Training deines Modells

> "Ich bin nicht gescheitert. Ich habe nur 10.000 Wege gefunden, die nicht funktionieren."
> — **Thomas Edison**, Erfinder

---

## Was du lernen wirst

- Wie Next-Token-Vorhersage Modelle das Schreiben lehrt
- Die 5-Schritte-Trainingsrezept, das alle neuronalen Netze antreibt
- Warum Verlustkurven zeigen, ob dein Modell lernt
- Wie du Überanpassung verhinderst und weisst, wann du stoppen solltest
- Speichern von Checkpoints, damit du niemals Fortschritt verlierst
- Der Nervenkitzel, dein Modell von Kauderwelsch zu Kohärenz verwandeln zu sehen

---

## Setup

Zuerst installieren wir die benötigten Pakete:

In [ ]:
# Getestete Paketversionen
!pip install -q torch==2.15.0 transformers==5.23.0 tqdm==4.70.1 numpy==2.3.5 matplotlib==3.10.7

In [ ]:
# ===== IMPORTS =====
import math
import urllib.request
from functools import partial
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm

# Gerät festlegen
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Verwendetes Gerät: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ===== REPRODUZIERBARKEIT =====
import random
import numpy as np

def set_seed(seed=42):
    """Setzt den Seed jeder Zufallsquelle, die wir verwenden."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)

## 1. Modellkomponenten aus Kapiteln 10-11

Zuerst holen wir das MiniGPT-Modell, das wir in den vorherigen Kapiteln erstellt haben.

In [ ]:
# ===== MULTI-HEAD ATTENTION (aus Kapitel 10) =====

class MultiHeadAttention(nn.Module):
    """Effiziente mehrköpfige Aufmerksamkeit (bündelt alle Köpfe zusammen)."""

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model muss durch num_heads teilbar sein"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        # bias=True entspricht GPT-2, damit wir später dessen vortrainierte Biases laden können
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)
        scores = scores / math.sqrt(d_k)

        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        attn_output = attn_weights @ V
        attn_output = attn_output.transpose(1, 2)
        attn_output = attn_output.reshape(batch, seq, d_model)

        output = self.out_proj(attn_output)
        return output, attn_weights

print("MultiHeadAttention definiert!")

In [ ]:
# ===== FEEDFORWARD-NETZWERK (aus Kapitel 10) =====

class FeedForward(nn.Module):
    """Positionsweises Feedforward-Netzwerk."""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x, approximate="tanh")  # GPT-2s GELU-Variante
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("FeedForward definiert!")

In [ ]:
# ===== TRANSFORMER-BLOCK (aus Kapitel 10) =====

class TransformerBlock(nn.Module):
    """Vollständiger Transformer-Block (Prä-Normalisierung wie GPT-2)."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)
        return x, attn_weights

print("TransformerBlock definiert!")

In [ ]:
# ===== GPT-KONFIGURATION (aus Kapitel 11) =====
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """Konfiguration für das MiniGPT-Modell."""
    vocab_size: int = 50257
    max_seq_len: int = 1024
    embed_dim: int = 768
    num_heads: int = 12
    num_layers: int = 12
    d_ff: int = 3072
    dropout: float = 0.1

    def __post_init__(self):
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}) muss durch num_heads ({self.num_heads}) teilbar sein"

print("GPTConfig definiert!")

In [ ]:
# ===== MINIGPT-MODELL (aus Kapitel 11) =====

class MiniGPT(nn.Module):
    """Ein minimales GPT-Sprachmodell."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # Einbettungen
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # Transformer-Blöcke
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # Finale Schichtnormalisierung und LM-Kopf
        self.ln_f = nn.LayerNorm(config.embed_dim)
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # Gewichtsverknüpfung
        self.lm_head.weight = self.token_embed.weight

        # Gewichte initialisieren
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        batch, seq = token_ids.shape
        device = token_ids.device

        tok_emb = self.token_embed(token_ids)
        positions = torch.arange(seq, device=device)
        pos_emb = self.pos_embed(positions)
        x = self.dropout(tok_emb + pos_emb)

        mask = torch.tril(torch.ones(seq, seq, device=device))

        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        if return_attention:
            return logits, attention_weights
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        Tokens generieren, die input_ids fortsetzen (Prompt ist in der Ausgabe enthalten).

        Args:
            input_ids: Prompt-Token-IDs (batch, seq)
            max_new_tokens: Wie viele neue Tokens generiert werden
            temperature: Sampling-Temperatur (nur verwendet, wenn do_sample=True)
            do_sample: Aus der Verteilung samplen; False = gieriges argmax
            eos_token_id: Früh stoppen, wenn jede Sequenz dieses Token generiert

        Returns:
            Token-IDs (batch, seq + neue Tokens)
        """
        self.eval()
        # Positions-Einbettungen decken nur max_seq_len Positionen ab
        limit = self.config.max_seq_len
        if input_ids.shape[1] + max_new_tokens > limit:
            raise ValueError(
                f"Prompt ({input_ids.shape[1]} Tokens) + max_new_tokens "
                f"({max_new_tokens}) überschreitet den {limit}-Token-Kontext "
                "des Modells. Kürze den Prompt oder senke max_new_tokens.")
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids

print("MiniGPT-Klasse definiert!")

## 2. Datensatz herunterladen

Wir verwenden TinyShakespeare - klein genug, um in Minuten zu trainieren.

In [ ]:
# TinyShakespeare herunterladen
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

with open("shakespeare.txt", "r") as f:
    text = f.read()

print(f"Datensatzgrösse: {len(text):,} Zeichen")
print(f"\nBeispiel:\n{text[:500]}")

## 3. Dataset und DataLoader erstellen

Beachte, was `create_lm_labels` NICHT tut: nichts maskieren. Pads maskieren wir in den Trainings- und Evaluationsschleifen über die Attention-Maske (`targets[attention_mask[:, 1:] == 0] = -100`), ein echtes EOS bleibt damit überwacht. GPT-2 paddet mit seinem EOS-Token (ID 50256): Eine wertbasierte Maske kann Padding nicht von einem echten Textende unterscheiden und würde jede echte EOS-Vorhersage stillschweigend löschen, dem Modell also beibringen, nie zu stoppen.


In [ ]:
def create_lm_labels(token_ids):
    """
    Verschobene Eingaben und Ziele für Sprachmodellierung erstellen.

    Args:
        token_ids: Token-IDs (batch, seq)

    Returns:
        inputs: (batch, seq-1), alle Tokens ausser dem letzten
        targets: (batch, seq-1), alle Tokens ausser dem ersten
    """
    inputs = token_ids[:, :-1]    # Alles ausser dem letzten Token
    targets = token_ids[:, 1:].clone()  # .clone(): die Schleifen markieren Pads direkt mit -100

    # Hier wird nichts maskiert: Pads werden in den Schleifen über die
    # Attention-Maske maskiert; ein echtes EOS (GPT-2 paddet mit eos, ID 50256)
    # bleibt damit überwacht.
    return inputs, targets


from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    """Einfacher Datensatz, der Textblöcke zurückgibt."""

    def __init__(self, text, chunk_size=512):
        self.chunks = []
        for i in range(0, len(text) - chunk_size + 1, chunk_size):
            self.chunks.append(text[i:i + chunk_size])

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]


def collate_fn(batch, tokenizer, max_length=128):
    """Tokenisiert und fügt Padding zu einem Batch von Textstrings hinzu."""
    encoded = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encoded["input_ids"], encoded["attention_mask"]


# Tokenizer laden (GPT-2 hat standardmässig kein Pad-Token; End-of-Text wiederverwenden)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Datensatz erstellen: Auf TinyShakespeare ergeben 512-Zeichen-Chunks im
# Schnitt ~156 GPT-2-Tokens; die max_length=128-Kürzung schneidet bei den
# meisten Chunks den Rest ab. Fürs Pretraining okay: Jeder Chunk bleibt
# dichtes Trainingssignal
dataset = TextDataset(text, chunk_size=512)
print(f"Anzahl der Chunks: {len(dataset)}")

In [ ]:
# Train/Val-Aufteilung erstellen
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Training: {len(train_dataset)}, Validierung: {len(val_dataset)}")

# DataLoaders erstellen
collate = partial(collate_fn, tokenizer=tokenizer, max_length=128)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate
)

# Einen Batch überprüfen
input_ids, attention_mask = next(iter(train_loader))
print(f"Batch input_ids Form: {input_ids.shape}")
print(f"Batches: {len(train_loader)} Training, {len(val_loader)} Validierung")
# Beide Teilmengen stammen aus demselben Werk und teilen Stil und Vokabular.
# Übung 1 trennt stattdessen ganze Dokumente vor dem Chunking.

## 4. Modell erstellen

In [ ]:
# Kleine Konfiguration für schnelles Training
config = GPTConfig(
    vocab_size=50257,
    max_seq_len=128,
    embed_dim=256,
    num_heads=4,
    num_layers=4,
    d_ff=1024,
    dropout=0.1
)

model = MiniGPT(config).to(device)
print(f"Parameter: {sum(p.numel() for p in model.parameters()):,}")

## 5. Der "Vorher"-Zustand: Untrainiertes Modell

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=1.0):
    """Text mit Temperaturkontrolle generieren."""
    device = next(model.parameters()).device
    token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # do_sample=True: Logits durch Temperatur teilen, dann samplen
    output_ids = model.generate(
        token_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(output_ids[0])


# Aus untrainiertem Modell generieren
print("VOR DEM TRAINING (zufällige Gewichte):")
print("="*50)
prompt = "The king"
print(f"Prompt: '{prompt}'")
print(f"Ausgabe: {generate(model, tokenizer, prompt)}")
print("\n(Zufälliges Kauderwelsch - das Modell hat noch nichts gelernt!)")


## 6. Training-Setup

In [ ]:
# Training-Hyperparameter
num_epochs = 3
learning_rate = 3e-4
warmup_steps = 100

total_steps = len(train_loader) * num_epochs
print(f"Gesamte Trainingsschritte: {total_steps}")

# Optimierer und Scheduler
# Parametergruppen-Helfer aus Kapitel 11
def get_parameter_groups(model, weight_decay=0.1):
    decay_params = []
    no_decay_params = []
    for name, param in model.named_parameters():
        if param.ndim < 2:
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    return [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ]

# Parametergruppen-Helfer aus Kapitel 11: Matrizen bekommen Weight Decay,
# Biases und LayerNorm-Vektoren nicht
optimizer = AdamW(get_parameter_groups(model, weight_decay=0.01),
                  lr=learning_rate)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

## 7. Training- und Evaluierungsfunktionen

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device, clip_norm=1.0):
    """Trainiert eine Epoche mit dem 5-Schritte-Rezept."""
    model.train()
    total_loss = 0

    progress = tqdm(dataloader, desc="Training")
    for input_ids, attention_mask in progress:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # Eingaben und Ziele erstellen (um eins verschoben), Pads mit -100 markiert
        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # Pads im Verlust ignorieren

        # ===== DAS 5-SCHRITTE-REZEPT =====
        optimizer.zero_grad(set_to_none=True)       # 1. Gradienten nullsetzen
        logits = model(inputs)                       # 2. Vorwärtsdurchlauf
        loss = F.cross_entropy(                      # 3. Verlust berechnen
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # reshape behandelt nicht-zusammenhängende Slices
            ignore_index=-100  # Padding-Positionen überspringen
        )
        loss.backward()                              # 4. Rückwärtsdurchlauf
        # Wichtig: Clippen NACH backward(), aber VOR step()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        optimizer.step()                             # 5. Gewichte aktualisieren
        scheduler.step()

        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader, device):
    """Evaluiert das Modell und berechnet die Perplexität."""
    model.eval()
    total_loss = 0
    total_tokens = 0

    for input_ids, attention_mask in dataloader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # Pads im Verlust ignorieren

        logits = model(inputs)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # reshape behandelt nicht-zusammenhängende Slices
            ignore_index=-100,
            reduction='sum'
        )

        # Nur Nicht-Padding-Tokens zählen, damit die Perplexität ehrlich ist
        total_loss += loss.item()
        total_tokens += (targets != -100).sum().item()

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity


def should_stop_early(val_losses, patience=3):
    """
    Prüfen, ob wir das Training stoppen sollten.

    Args:
        val_losses: Liste der Validierungsverluste pro Epoche
        patience: Wie viele ansteigende Epochen wir tolerieren

    Returns:
        True, wenn wir stoppen sollten
    """
    if len(val_losses) < patience + 1:
        return False

    # Prüfen, ob die letzten 'patience' Verluste alle ansteigen
    recent = val_losses[-(patience + 1):]
    for i in range(1, len(recent)):
        if recent[i] <= recent[i-1]:
            return False

    return True


In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, path):
    """Speichert einen Training-Checkpoint, inklusive RNG-Zustände für exaktes Fortsetzen."""
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        # RNG-Zustände: Dropout und Shuffling müssen dieselbe Folge fortsetzen
        'rng_python': random.getstate(),
        'rng_numpy': np.random.get_state(),
        'rng_torch': torch.get_rng_state(),
        'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        'rng_mps': torch.mps.get_rng_state() if torch.backends.mps.is_available() else None,
    }, path)
    print(f"Checkpoint gespeichert: {path}")


def load_checkpoint(model, optimizer, scheduler, path, device):
    """
    Training-Checkpoint laden und RNG-Zustände wiederherstellen.

    Returns:
        epoch: Die Epoche, ab der fortgesetzt wird
        train_loss: Letzter Trainingsverlust
        val_loss: Letzter Validierungsverlust
    """
    checkpoint = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # RNG-Zustände müssen CPU-Tensoren sein: map_location=device verschiebt sie sonst
    random.setstate(checkpoint['rng_python'])
    np.random.set_state(checkpoint['rng_numpy'])
    torch.set_rng_state(checkpoint['rng_torch'].cpu())
    if checkpoint['rng_cuda'] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in checkpoint['rng_cuda']])
    if checkpoint['rng_mps'] is not None and torch.backends.mps.is_available():
        torch.mps.set_rng_state(checkpoint['rng_mps'].cpu())

    print(f"Checkpoint geladen aus Epoche {checkpoint['epoch']}")

    return checkpoint['epoch'], checkpoint['train_loss'], checkpoint['val_loss']

Ein Trade-off: `weights_only=True` verweigert beim Laden das Ausführen beliebigen Pickle-Codes, die richtige Gewohnheit für Checkpoints aus unvertrauenswürdigen Quellen. NumPys RNG-Zustand enthält aber NumPy-Skalare, die der sichere Loader ablehnt, deshalb nutzt diese exakt fortsetzbare Version `weights_only=False`. Lade so nur Checkpoints, die du selbst erstellt hast (oder denen du ausdrücklich vertraust); für alles Heruntergeladene bevorzuge `weights_only=True` und akzeptiere, dass die RNG-Zustände nicht wiederhergestellt werden. Zwei weitere Bedingungen für bitgenaues Fortsetzen: den Scheduler mit demselben Gesamtschritt-Horizont wie der Originallauf neu bauen und die Batches in derselben Reihenfolge verbrauchen.

## 8. Trainingsschleife

In [ ]:
# Training!
train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoche {epoch + 1}/{num_epochs}")
    print(f"{'='*50}")

    # Trainieren
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    train_losses.append(train_loss)

    # Evaluieren
    val_loss, perplexity = evaluate(model, val_loader, device)
    val_losses.append(val_loss)

    print(f"\nTraining-Verlust: {train_loss:.4f}")
    print(f"Val-Verlust:      {val_loss:.4f}")
    print(f"Perplexität:      {perplexity:.1f}")

    # Besten Checkpoint speichern
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, scheduler, epoch,
                       train_loss, val_loss, "best_model.pt")

print("\nTraining abgeschlossen!")
print(f"Bester Validierungsverlust: {best_val_loss:.4f}")

## 9. Verlustkurven zeichnen

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'b-o', label='Training-Verlust')
plt.plot(epochs, val_losses, 'r-o', label='Validierungs-Verlust')
plt.xlabel('Epoche')
plt.ylabel('Verlust')
plt.title('Training- und Validierungsverlust')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. Die Belohnung: Textgenerierung!

In [ ]:
# Bestes Modell laden (eigener Checkpoint, daher ist weights_only=False hier sicher;
# die enthaltenen RNG-Zustände brauchen den vollen Loader, siehe Checkpoint-Abschnitt)
checkpoint = torch.load("best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Checkpoint geladen aus Epoche {checkpoint['epoch'] + 1}")
print(f"Validierungsverlust: {checkpoint['val_loss']:.4f}")

In [ ]:
print("\n" + "="*60)
print("NACH DEM TRAINING (auf Shakespeare):")
print("="*60)

prompts = [
    "The king",
    "To be or not to be",
    "Friends, Romans, countrymen",
    "All the world's a stage"
]

for prompt in prompts:
    output = generate(model, tokenizer, prompt, max_new_tokens=40, temperature=0.8)
    print(f"\nPrompt: '{prompt}'")
    print(f"Ausgabe: {output}")
    print("-"*40)

## 11. Vorher-Nachher-Vergleich

In [ ]:
# Neues untrainiertes Modell zum Vergleich erstellen
set_seed(123)  # Anderer Seed für unterschiedliche zufällige Gewichte
untrained_model = MiniGPT(config).to(device)
untrained_model.eval()

prompt = "The fair maiden"

print("="*60)
print(f"Prompt: '{prompt}'")
print("="*60)
print("\nVORHER (untrainiert):")
print(generate(untrained_model, tokenizer, prompt, max_new_tokens=30))
print("\nNACHHER (auf Shakespeare trainiert):")
print(generate(model, tokenizer, prompt, max_new_tokens=30, temperature=0.8))
print("\nGleiche Architektur. Gleicher Code. Training macht den ganzen Unterschied!")

## 12. Temperaturvergleich

In [ ]:
prompt = "The noble lord"

print(f"Prompt: '{prompt}'\n")
for temp in [0.5, 0.8, 1.0, 1.5]:
    output = generate(model, tokenizer, prompt, max_new_tokens=30, temperature=temp)
    print(f"Temperatur {temp}:")
    print(f"  {output}")
    print()

## Zusammenfassung

**Was wir gebaut haben:**

1. **Label-Verschiebung** für Next-Token-Vorhersage
2. **DataLoaders**, die Text effizient batchen und tokenisieren
3. **Das 5-Schritte-Trainingsrezept**: zero_grad → forward → loss → backward → step
4. **Evaluierung** mit Validierungsverlust und Perplexität
5. **Checkpointing** zum Speichern und Fortsetzen des Trainings
6. **Textgenerierung** mit Temperaturkontrolle

**Schlüsselkonzepte:**

- Training ist eine Feedback-Kontrollschleife: Fehler messen, Gewichte anpassen, wiederholen
- Kreuzentropie-Verlust bestraft zuversichtliche falsche Vorhersagen stärker als unsichere
- Überanpassung = Auswendiglernen von Trainingsdaten (Training-Verlust ↓, Val-Verlust ↑)
- Perplexität misst, wie "überrascht" das Modell ist - niedriger ist besser
- Temperatur kontrolliert die Generierungsvielfalt

**Weiter:** Kapitel 13 wird dir beibringen, dieses Modell für spezifische Aufgaben fein abzustimmen!

## Übungen

Übung 1 verwendet wie im Buch den Gartenkorpus. Die Notebook-Übungen 2 und 3 variieren Epochenzahl und Modellgrösse; den Lernratenvergleich und den Neustart vom Checkpoint findest du in den Buchübungen 2 und 4.

### Übung 1: Einen eigenen JSONL-Korpus verwenden

Drei kurze Kapitel-7-Datensätze reichen nicht für 512-Zeichen-Chunks. Der folgende Gartenkorpus zeigt denselben Ladeweg mit 16 Trainingsdokumenten und vier separaten Validierungstexten.

In [ ]:
import json

base_paragraphs = [
    "Der Gemeinschaftsgarten öffnet am Wochenende bei Sonnenaufgang. Die Mitglieder "
    "giessen zuerst die Tomaten und schauen dann nach den Kompostkisten. Neue "
    "Freiwillige melden sich beim Werkzeugschuppen an, wo Handschuhe und "
    "Pflanzkellen in nummerierten Reihen hängen. Alle teilen die Ernte am "
    "Sonntagstisch beim Tor.",
    "Regen sammelt sich in den zwei Fässern neben dem Gewächshaus. In trockenen "
    "Wochen tragen wir Eimer über die mulchbedeckten Wege, vorbei an den Bohnen "
    "und den hohen Sonnenblumen. Der Giessplan hängt an einer Tafel, und eine "
    "Schicht zu tauschen braucht eine Linie Kreide und ein kurzes Hallo.",
    "Die Jungpflanzen starten im März drinnen, unter Lampen im alten "
    "Topfraum. Wir beschriften jedes Tray mit der Sorte und dem Aussaatdatum. "
    "Wenn der letzte Frost vorbei ist, ziehen die Trays für eine Woche "
    "Abhärtung nach draussen, bevor sie ein eigenes Beet bekommen.",
    "Das Kompostrezept ist einfach: drei Teile trockenes Laub auf einen Teil "
    "Küchenabfälle, alle zwei Wochen umgesetzt. An kalten Morgen steigt Dampf "
    "aus einem gesunden Haufen auf. Fertiger Kompost riecht nach Waldboden und "
    "nährt die Kürbisbeete der nächsten Saison.",
    "Der Herbst bedeutet Samen sammeln und Beete schliessen. Wir sammeln "
    "Bohnen- und Salatssamen in Papiertüten, trocknen sie auf dem Regal über "
    "der Werkbank und schreiben das Jahr auf jede Tüte. Das Tor bekommt seinen "
    "Winterriegel, und der Garten schläft bis Februar.",
]
val_paragraphs = [
    "Am Samstag bauen wir ein Spalier für die Erbsen. Zwei Mitglieder halten die "
    "Pfosten, während die anderen Schnüre zwischen den Querlatten spannen. Die "
    "unterste Reihe sitzt knapp über der Erde, damit auch die jüngsten Triebe "
    "ohne Hilfe einen Halt finden.",
    "Neben dem Tor bleibt ein Streifen für Wildblumen frei. Dort zählen wir an "
    "drei warmen Nachmittagen die Hummeln und tragen die Zahlen ins Gartenbuch "
    "ein. Beim Mähen lassen wir immer eine Hälfte stehen, bis auf der anderen "
    "Seite wieder genug Blüten offen sind.",
    "Heute prüfen wir die Kartoffeln auf dem hinteren Beet. Unter einer welken "
    "Staude liegen schon fünf grosse Knollen. Wir graben nur die reifen Reihen "
    "aus, lassen die Schalen im Schatten trocknen und verteilen die Ernte erst "
    "am Abend auf die mitgebrachten Körbe.",
    "Vor dem ersten Frost räumen wir das kleine Gewächshaus auf. Die leeren "
    "Töpfe werden gebürstet und nach Grösse gestapelt. An der Tür hängt jetzt "
    "ein Zettel mit den fehlenden Werkzeugen, damit beim nächsten Arbeitstag "
    "niemand lange nach einer Schaufel suchen muss.",
]
records = []
for i in range(20):
    body = base_paragraphs[i % 5] if i < 16 else val_paragraphs[i - 16]
    text = f"Eintrag {i + 1}. " + body
    records.append({"text": text, "split": "train" if i < 16 else "val",
                    "length": len(text)})
with open("my_corpus.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")
print(f"{len(records)} Records in my_corpus.jsonl geschrieben")

def load_split_texts(path, split):
    """Verbinde die Textfelder aller Records, die mit diesem Split getaggt sind."""
    texts = []
    with open(path) as f:
        for line in f:
            record = json.loads(line)
            if record.get("split") == split:
                texts.append(record["text"])
    return "\n".join(texts)

# Zuerst der ehrliche Check am Demo-Artefakt aus Kapitel 7:
from pathlib import Path
if Path("train.jsonl").is_file():
    demo = TextDataset(load_split_texts("train.jsonl", "train"), chunk_size=512)
    print(f"Kapitel-7-Demo-Korpus: {len(demo)} Chunks (zu klein zum Trainieren)")

else:
    print("Für den optionalen Demo-Vergleich: train.jsonl aus Kapitel 7 hierher übertragen.")

# Jetzt das Fixture (tausch deinen eigenen grösseren Export genauso ein):
train_dataset = TextDataset(load_split_texts("my_corpus.jsonl", "train"), chunk_size=512)
val_dataset = TextDataset(load_split_texts("my_corpus.jsonl", "val"), chunk_size=512)
print(f"Train-Chunks: {len(train_dataset)}, Val-Chunks: {len(val_dataset)}")

assert len(TextDataset('x' * 1024)) == 2 and len(TextDataset('x' * 1023)) == 1
train_bodies, val_bodies = ({r['text'].split('. ', 1)[1] for r in records if r['split'] == split}
                            for split in ('train', 'val'))
assert len(val_bodies) == 4 and val_bodies.isdisjoint(train_bodies)
assert (len(train_dataset), len(val_dataset)) == (9, 2)

### Übung 2: Mehr Epochen

In [ ]:
# DEIN CODE HIER
# Trainiere für 5-10 Epochen statt 3
# Verbessert sich das Modell weiter?
# Siehst du Anzeichen von Überanpassung?

### Übung 3: Modellgrössen-Vergleich

In [ ]:
# DEIN CODE HIER
# Erstelle ein kleineres Modell (2 Schichten, 128 Dimensionen)
# Erstelle ein grösseres Modell (6 Schichten, 384 Dimensionen)
# Vergleiche Trainingsgeschwindigkeit und finale Perplexität